# Project links

**Github repository**
https://github.com/phkhle/IND320/tree/project3

**Streamlitt app**
https://ind320-h2026.streamlit.app/


In [1]:
import geopandas as gpd

# Download Euro, USD currency exchange rates

# Streamlit application 

## Structure and navigation

For Project 4, the Streamlit dashboard is reorganised according to the purpose of each page rather than only the data source.

The application is divided into three main sections:

- Explore Data: 
    - Interactive Map
    - Reservoir Statistics 
    - Electricity Transfer
    - Currency Exchange.

- Analysis: 
    - STL and Spectrogram
    - Weather Anomalies
    - Sliding Window Correlation.

- Forecasting: Reservoir Filling Forecast.

A separate Home page provides an overview of the dashboard and its data sources.

The navigation was implemented using ``st.navigation()`` and ``st.Page()``. Pages can then be grouped into logical sections.

Shared geographical selections will be stored using ``st.session_state``. Users will be able to select an area on the interactive map and reuse that selection in other analyses.



## Interactive map (XXX)

## Create GeoJSON

Create geographic file for the electricity pricing areas (EL)

In [2]:
# Load electricity price areas
price_areas = gpd.read_file(
    "../data/shapefiles/Budområder.shp"
)

# Convert to WGS84 coordinates
price_areas = price_areas.to_crs(epsg=4326)

# Standardize region names
price_areas["region"] = (
    price_areas["budomr"]
    .str.replace(" ", "", regex=False)
)

# Save as GeoJSON
price_areas[["region", "geometry"]].to_file(
    "../data/price_areas.geojson",
    driver="GeoJSON"
)

print("GeoJSON for electricity price areas created successfully!")

GeoJSON for electricity price areas created successfully!


Create geographic file for the precipitation areas

In [3]:
# Load rain areas
regions = gpd.read_file("../data/shapefiles/NVE_region.shp", encoding = 'utf-8')

# Map NVE administrative regions to precipitation regions
region_mapping = {
    "RS": "Øst",
    "RØ": "Øst",
    "RV": "Vest",
    "RM": "Nord",
    "RN": "Nord"
}

# Assign precipitation region names
precipitation_regions = regions.copy()
precipitation_regions["region"] = precipitation_regions["REGION"].map(region_mapping)

# Merge polygons belonging to the same precipitation region
precipitation_regions = precipitation_regions.dissolve(
    by="region",
    as_index=False
)

# Convert precipitation regions to WGS84
weather_regions_geojson = precipitation_regions.to_crs(epsg=4326)

# Keep only region names and geometry
weather_regions_geojson = weather_regions_geojson[
    ["region", "geometry"]
]

# Export as GeoJSON
weather_regions_geojson.to_file(
    "../data/weather_regions.geojson",
    driver="GeoJSON"
)

print("Weather regions GeoJSON created successfully!")

Weather regions GeoJSON created successfully!


Create geographic file for the water areas (river basin areas - VASS)

In [4]:
# Load water areas
water_regions = gpd.read_file("../data/shapefiles/Vassdragsomrade.shp", encoding = 'utf-8')
# Convert to WGS84 coordinates
water_regions = water_regions.to_crs(epsg=4326)


# Standardize region names
water_regions["region"] = (
    water_regions["vassdragso"]
    .str.replace(" ", "", regex=False)
)

# Save as GeoJSON
water_regions[["region", "geometry"]].to_file(
    "../data/water_areas.geojson",
    driver="GeoJSON"
)


In [5]:
import geopandas as gpd

water_regions = gpd.read_file("../data/water_areas.geojson")

print("Antall områder:", len(water_regions))
print("Kolonner:", water_regions.columns.tolist())

display(water_regions[["region"]])

Antall områder: 205
Kolonner: ['region', 'geometry']


,region
0,196
1,191
2,190
3,189
4,315
...,...
200,005
201,004
202,003
203,002


# Forecasting

In [12]:
import sys
from pathlib import Path

# Get the parent directory, since the notebook is located one folder below the project root
project_root = Path.cwd().parent
# Add the project root to Python's module search path
# This allows Python to find and import modules from src/
sys.path.insert(0, str(project_root))

from src.NVE_API import get_nve_data

from src.forecasting import forecast_reservoir
# Retrieve reservoir data from NVE API
df_nve = get_nve_data()

# Select NO3
df_region = df_nve[
    (df_nve["omrType"] == "EL") &
    (df_nve["omrnr"] == 3)
].copy()

results = forecast_reservoir(
    df=df_region,
    n_lags=4,
    test_weeks=52
)

results.head()

,date,actual,predicted
0,2025-11-23,0.851878,0.866210
1,2025-11-30,0.824069,0.842509
2,2025-12-07,0.795395,0.804163
3,2025-12-14,0.774006,0.767300
4,2025-12-21,0.766409,0.748065


In [13]:
from src.NVE_API import get_nve_data

df_nve = get_nve_data()

# Select NO3
df_region = df_nve[
    (df_nve["omrType"] == "EL") &
    (df_nve["omrnr"] == 3)
].copy()

df_region = df_region.sort_values("dato_Id")

print("First observation:", df_region["dato_Id"].min())
print("Last observation:", df_region["dato_Id"].max())
print("Number of weeks:", len(df_region))

First observation: 1995-01-08 00:00:00
Last observation: 2026-11-15 00:00:00
Number of weeks: 1663


In [14]:
from src.forecasting import forecast_reservoir

results = forecast_reservoir(
    df=df_region,
    n_lags=4,
    test_weeks=52
)

print(results.head())

        date    actual  predicted
0 2025-11-23  0.851878   0.866210
1 2025-11-30  0.824069   0.842509
2 2025-12-07  0.795395   0.804163
3 2025-12-14  0.774006   0.767300
4 2025-12-21  0.766409   0.748065


In [15]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(
    results["actual"],
    results["predicted"]
)

rmse = np.sqrt(mean_squared_error(
    results["actual"],
    results["predicted"]
))

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

MAE: 0.0179
RMSE: 0.0218


In [16]:
# Baseline: previous week's filling
df_region["baseline"] = df_region["fyllingsgrad"].shift(1)

baseline_df = results.merge(
    df_region[["dato_Id", "baseline"]],
    left_on="date",
    right_on="dato_Id",
    how="left"
)

baseline_mae = mean_absolute_error(
    baseline_df["actual"],
    baseline_df["baseline"]
)

baseline_rmse = np.sqrt(mean_squared_error(
    baseline_df["actual"],
    baseline_df["baseline"]
))

print(f"River MAE:    {mae * 100:.2f} pp")
print(f"River RMSE:   {rmse * 100:.2f} pp")
print(f"Baseline MAE: {baseline_mae * 100:.2f} pp")
print(f"Baseline RMSE:{baseline_rmse * 100:.2f} pp")

River MAE:    1.79 pp
River RMSE:   2.18 pp
Baseline MAE: 2.44 pp
Baseline RMSE:3.07 pp
